# Check Yourself: Coding Practice

Welcome to the interactive coding environment! Use this notebook to solve assignments and submit them for evaluation.

## 1. Authentication
Enter your credentials to login to the platform.

In [ ]:
import requests
import json
import time
from IPython.display import display, Markdown, clear_output

BASE_URL = "http://localhost:8000"
EMAIL = "test@example.com"
PASSWORD = "password123"

def login(email, password):
    resp = requests.post(f"{BASE_URL}/auth/login", data={"username": email, "password": password})
    if resp.status_code == 200:
        token = resp.json()["access_token"]
        print("Successfully logged in!")
        return {"Authorization": f"Bearer {token}"}
    else:
        print("Login failed:", resp.text)
        return None

headers = login(EMAIL, PASSWORD)

## 2. Fetch Assignments
Select a coding assignment from the list below.

In [ ]:
def list_assignments():
    resp = requests.get(f"{BASE_URL}/assignments", headers=headers)
    if resp.status_code == 200:
        assignments = resp.json()
        coding_tasks = [a for a in assignments if a["assignment_type"] == "coding"]
        for i, task in enumerate(coding_tasks):
            print(f"[{i}] {task['title']} (ID: {task['id']})")
        return coding_tasks
    return []

tasks = list_assignments()

## 3. View Assignment Details
Pick a task index (e.g., 0) to see the problem statement.

In [ ]:
task_index = 0
current_task_id = tasks[task_index]['id']

resp = requests.get(f"{BASE_URL}/assignments/{current_task_id}", headers=headers)
task_detail = resp.json()

display(Markdown(f"### {task_detail['title']}"))
display(Markdown(task_detail['description']))
print(f"Target function: {task_detail.get('function_name', 'Not specified')}")

## 4. Solve and Submit
Write your solution in the cell below and run it to submit.

In [ ]:
## WRITE YOUR CODE HERE
user_code = """
def solve(n):
    return sum(range(n + 1))
"""

def submit(code):
    payload = {
        "assignment_id": current_task_id,
        "code": code,
        "language": "python"
    }
    resp = requests.post(f"{BASE_URL}/coding-assignments/submit", json=payload, headers=headers)
    if resp.status_code == 202:
        sub_id = resp.json()["id"]
        print(f"Submission successful! ID: {sub_id}")
        return sub_id
    else:
        print("Submission failed:", resp.text)
        return None

submission_id = submit(user_code)

## 5. View Results
Wait a few seconds for the sandbox to evaluate your code.

In [ ]:
def poll_result(sub_id):
    print("Evaluating...", end="")
    for _ in range(10):
        time.sleep(2)
        resp = requests.get(f"{BASE_URL}/coding-assignments/submissions/{sub_id}", headers=headers)
        res = resp.json()
        if res["status"] not in ["pending", "running"]:
            clear_output(wait=True)
            print(f"Status: {res['status']}")
            print(f"Score: {res['score']}% ({res['passed_cases']}/{res['total_cases']} cases passed)")
            if res.get('error_message'):
                print(f"Error: {res['error_message']}")
            return
        print(".", end="")
    print("\nTimed out waiting for result.")

if submission_id:
    poll_result(submission_id)